In [1]:
!pip install opencv-python

import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm import tqdm

In [2]:
class SignLanguageDataset(Dataset):
    
    def __init__(self, root_dir, num_frames=20):
        self.samples = []
        self.total_videos = 0
        self.num_frames = num_frames
        self.bad_videos = 0
    
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((224, 224)),
        ])
        
        for group in os.listdir(root_dir):
            group_path = os.path.join(root_dir, group)
            if not os.path.isdir(group_path):
                continue
            
            for inner in os.listdir(group_path):
                inner_path = os.path.join(group_path, inner)
                if not os.path.isdir(inner_path):
                    continue
                
                for person in os.listdir(inner_path):
                    person_path = os.path.join(inner_path, person)
                    if not os.path.isdir(person_path):
                        continue
                    
                    for sentence in os.listdir(person_path):
                        sentence_path = os.path.join(person_path, sentence)
                        if not os.path.isdir(sentence_path):
                            continue
                        
                        for file in os.listdir(sentence_path):
                            if file.endswith(('.mp4', '.avi', '.mov')):
                                video_path = os.path.join(sentence_path, file)
                                
                                label = sentence.lower()
                                self.samples.append((video_path, label))
                                self.total_videos += 1
    
        all_labels = [s[1] for s in self.samples]
        self.labels = list(set(all_labels))
        self.label_map = {label: idx for idx, label in enumerate(self.labels)}
            
    def __len__(self):
        return len(self.samples)
    
    def load_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        
        cap.release()
        
        if len(frames) == 0:
            self.bad_videos += 1   # 🔥 increment counter
            return torch.zeros((self.num_frames, 3, 224, 224))
        
        if len(frames) < self.num_frames:
            frames = frames + [frames[-1]] * (self.num_frames - len(frames))
        
        idxs = np.linspace(0, len(frames)-1, self.num_frames).astype(int)
        frames = [frames[i] for i in idxs]
        
        frames = [self.transform(frame) for frame in frames]
        
        return torch.stack(frames)
    
    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        
        frames = self.load_video(video_path)
        label = self.label_map[label]
        
        return frames, label

In [3]:
class CNNLSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        # CNN backbone
        resnet = models.resnet50(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        
        # Freeze CNN
        for param in self.cnn.parameters():
            param.requires_grad = False
        
        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=256,
            num_layers=1,
            batch_first=True
        )
        
        self.fc = nn.Linear(256, num_classes)
    
    def forward(self, x):
        B, T, C, H, W = x.shape
        
        x = x.view(B*T, C, H, W)
        features = self.cnn(x)  # (B*T, 2048, 1, 1)
        features = features.view(B, T, 2048)
        
        _, (h, _) = self.lstm(features)
        
        out = self.fc(h[-1])
        return out

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = SignLanguageDataset("/kaggle/input/datasets/belovedorange/nlp-dataset")

# from torch.utils.data import Subset

# subset_size = int(0.01 * len(dataset))  # 1%
# indices = list(range(subset_size))

# dataset = Subset(dataset, indices)

from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# print("Total videos found:", dataset.total_videos)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

model = CNNLSTM(num_classes=len(dataset.labels)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 208MB/s]


In [5]:
# import os

# root = "/kaggle/input/datasets/belovedorange/nlp-dataset"

# groups = os.listdir(root)
# print("Total groups:", len(groups))
# print(groups)

# print()

# group_path = os.path.join(root, groups[0])
# print("Group:", group_path)

# print("\nInside group:")
# print(os.listdir(group_path))

# person_folders = os.listdir(group_path)
# person_path = os.path.join(group_path, person_folders[0])

# print("\nPerson folder:", person_path)
# print(os.listdir(person_path))

# sentence_folders = os.listdir(person_path)
# sentence_path = os.path.join(person_path, sentence_folders[0])

# print("\nSentence folder:", sentence_path)
# print(os.listdir(sentence_path))

# video_files = os.listdir(sentence_path)

# print("\nVideos inside sentence:")
# print(video_files)
# print("Number of videos:", len(video_files))

# deep_path = os.path.join(sentence_path, video_files[0])

# print("Deep path:", deep_path)
# print("Inside:", os.listdir(deep_path))

In [6]:
epochs = 7
best_val_acc = 0
patience = 2
counter = 0

for epoch in range(epochs):
    
    # ===== TRAIN =====
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for videos, labels in tqdm(train_loader):
        videos = videos.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(videos)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    
    
    # ===== VALIDATION =====
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device)
            labels = labels.to(device)
            
            outputs = model(videos)
            
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    
    # ✅ ADD THIS LINE
    val_acc = (val_correct / val_total) * 100
    
    
    # 🔥 EARLY STOPPING BLOCK (PUT HERE)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0
        
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Best model saved!")
    else:
        counter += 1
        print(f"No improvement. Counter: {counter}/{patience}")
    
    if counter >= patience:
        print("⛔ Early stopping triggered")
        break
    
    
    # ===== PRINT =====
    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss / len(train_loader):.4f}")
    print(f"Train Accuracy: {(train_correct / train_total) * 100:.2f}%")
    print(f"Val Accuracy: {val_acc:.2f}%")


print("\n===== DATASET STATS =====")
print("Total videos:", dataset.total_videos)
print("Bad videos encountered:", dataset.bad_videos)
print("Total classes:", len(dataset.labels))
print("Sample labels:", dataset.labels[:5])

if dataset.total_videos > 0:
    print("Percentage discarded:", 
          (dataset.bad_videos / dataset.total_videos) * 100, "%")

100%|██████████| 497/497 [58:44<00:00,  7.09s/it]


✅ Best model saved!

Epoch 1
Train Loss: 4.9407
Train Accuracy: 1.46%
Val Accuracy: 1.81%


100%|██████████| 497/497 [57:16<00:00,  6.91s/it]


✅ Best model saved!

Epoch 2
Train Loss: 4.8040
Train Accuracy: 2.31%
Val Accuracy: 2.21%


100%|██████████| 497/497 [57:31<00:00,  6.94s/it]


...

✅ Best model saved!

Epoch 4
Train Loss: 4.5881
Train Accuracy: 4.28%
Val Accuracy: 7.65%


100%|██████████| 497/497 [57:21<00:00,  6.92s/it]


No improvement. Counter: 1/2

Epoch 5
Train Loss: 4.4565
Train Accuracy: 6.14%
Val Accuracy: 6.84%


100%|██████████| 497/497 [57:28<00:00,  6.94s/it]


No improvement. Counter: 2/2
⛔ Early stopping triggered

===== DATASET STATS =====
Total videos: 2485
Bad videos encountered: 0
Total classes: 157
Sample labels: ['how can i help you', 'i am bored', ' you are bad', 'i am suffering from fever', 'pour some more water in the glass']
Percentage discarded: 0.0 %
